# Shift-Share (Bartik) Designs: Shares, Shocks, and Identification

**Econometrics Notebook Library · v0.1.0**

## Intuition

A shift-share instrument has the form

$$
z_i=\sum_s s_{is}g_s,
$$

where $s_{is}$ is region $i$'s baseline exposure to sector $s$ and $g_s$ is a sector-level shock.

The same algebra can support different research designs:

- **Share-exogeneity view:** identification comes from the exposure shares being as-good-as-random conditional on controls.
- **Shock-exogeneity view:** identification comes from the shocks being quasi-random, with many sufficiently dispersed shocks and predetermined exposure mapping.

Those stories imply different diagnostics.

References: [Goldsmith-Pinkham, Sorkin & Swift, AER (2020)](https://doi.org/10.1257/aer.20181047) · [Borusyak, Hull & Jaravel, ReStud (2022)](https://doi.org/10.1093/restud/rdab030).

## Shock-level equivalence

In the no-controls, no-intercept case,

$$
\hat\beta_{SSIV}=\frac{\sum_i z_i y_i}{\sum_i z_i x_i}
=\frac{\sum_s g_s\sum_i s_{is}y_i}{\sum_s g_s\sum_i s_{is}x_i}.
$$

Let $S_s=\sum_i s_{is}$ and $\bar y_s=(\sum_i s_{is}y_i)/S_s$. Then

$$
\hat\beta_{SSIV}=\frac{\sum_s S_sg_s\bar y_s}{\sum_s S_sg_s\bar x_s}.
$$

This shock-level representation is central to the quasi-experimental shock-exogeneity interpretation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4.5)
pd.set_option("display.max_columns", 30)

In [ ]:
from econnotes.core import (
    simulate_shift_share, iv_ratio, shock_level_iv_equivalent,
    effective_number_of_shocks
)

sim = simulate_shift_share(n_regions=220, n_sectors=30, seed=222)
beta_region = iv_ratio(sim["y"], sim["x"], sim["z"])
beta_shock = shock_level_iv_equivalent(sim["y"], sim["x"], sim["shares"], sim["shocks"])
{"true_beta":sim["beta"], "region_level_SSIV":beta_region, "shock_level_equivalent":beta_shock,
 "effective_number_of_shocks":effective_number_of_shocks(sim["shares"])}

## Concentration diagnostic

Even with 30 nominal sectors, exposure can be concentrated. A simple effective-number-of-shocks measure is

$$N_{eff}=1/\sum_s q_s^2,$$

where $q_s$ is sector $s$'s share of total exposure. This is not a complete inference diagnostic, but it immediately reveals whether “many shocks” is mostly fiction.

In [ ]:
expo = sim["shares"].sum(axis=0)
q = expo/expo.sum()
fig, ax = plt.subplots()
ax.bar(np.arange(len(q)), np.sort(q)[::-1])
ax.set(xlabel="Sector rank", ylabel="Exposure share", title="Shock exposure concentration");

## Common failure

A shift-share variable is not automatically exogenous because its shocks are “national.” If local outcomes mechanically contribute to measured national shocks, own-observation contamination can create endogeneity. Leave-one-out shock construction may be essential. Likewise, endogenous baseline shares invalidate the share-exogeneity story even when the shocks look clean.

## Researcher failure checklist

- State whether identification is from shares, shocks, or a different design argument.
- Report shock exposure concentration and influential sectors.
- Use predetermined shares and defend the baseline date.
- Construct leave-one-out shocks when local outcomes enter aggregate shocks.
- Residualize consistently when controls are present; the simple shock-level algebra above is a stripped-down identity, not a full empirical recipe.